In [1]:
import glob, os, shutil
src = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/VERSION", recursive=True)
       if open(p).read().strip().startswith("v6")]
assert src, "v6 code not found: attach the dataset with unlearning_tofu_experiment_v6.zip"
dst = "/kaggle/working/unlearning-project"
shutil.rmtree(dst, ignore_errors=True); shutil.copytree(src[0], dst); os.chdir(dst)
!pip install -q rouge-score bitsandbytes
os.makedirs("results/v6", exist_ok=True)
!python env_capture.py > results/v6/environment_run14.txt
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-9966ec69-d4eb-89fe-dcf7-f4bb146fb81c)
GPU 1: Tesla T4 (UUID: GPU-4c94d009-2754-bffd-d03d-66aea967f92a)


In [2]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF token loaded")
except Exception:
    print("no HF token (fine if the checkpoints are public)")
from huggingface_hub import HfApi
ids = [m.id for m in HfApi().list_models(author="open-unlearning", search="unlearn_tofu_Llama-3.2-1B-Instruct_forget10_", limit=2000)]
print(len(ids), "checkpoints found; first:", ids[:3])

no HF token (fine if the checkpoints are public)
428 checkpoints found; first: ['open-unlearning/unlearn_tofu_Llama-3.2-1B-Instruct_forget10_AltPO_lr5e-05_beta0.1_alpha1_epoch10', 'open-unlearning/unlearn_tofu_Llama-3.2-1B-Instruct_forget10_AltPO_lr5e-05_beta0.5_alpha2_epoch10', 'open-unlearning/unlearn_tofu_Llama-3.2-1B-Instruct_forget10_NPO_lr2e-05_beta0.5_alpha1_epoch10']


In [3]:
%cd /kaggle/working/unlearning-project
!MODEL3=unsloth/Llama-3.2-1B-Instruct HF_HOME=/tmp/hf CKPT_DIR=/tmp/ckpt python experiments_v6.py run14

/kaggle/working
README.md: 100%|███████████████████████████| 4.14k/4.14k [00:00<00:00, 11.9MB/s]
full.json: 100%|███████████████████████████| 1.08M/1.08M [00:00<00:00, 32.6MB/s]
Generating train split: 100%|█████| 4000/4000 [00:00<00:00, 88458.50 examples/s]
full                          4000 rows  columns=['question', 'answer']
retain90.json: 100%|██████████████████████████| 963k/963k [00:00<00:00, 138MB/s]
Generating train split: 100%|████| 3600/3600 [00:00<00:00, 523633.46 examples/s]
retain90                      3600 rows  columns=['question', 'answer']
forget10.json: 100%|█████████████████████████| 115k/115k [00:00<00:00, 76.7MB/s]
Generating train split: 100%|██████| 400/400 [00:00<00:00, 114099.67 examples/s]
forget10                       400 rows  columns=['question', 'answer']
forget10_perturbed.json: 100%|████████████████| 675k/675k [00:00<00:00, 148MB/s]
Generating train split: 100%|███████| 400/400 [00:00<00:00, 77457.14 examples/s]
forget10_perturbed             400 rows

In [4]:
%cd /kaggle/working/unlearning-project
!zip -qr /kaggle/working/results_v6_run14.zip results/v6/run14 results/v6/environment_run14.txt logs/v6

/kaggle/working
